# Exploración de archivos NDJSON de Reddit - versión 2

Notebook educativo en dos partes para trabajar con los datos de Reddit de la tesis.

**Parte 1 - Fundamentos** (secciones 1 a 9): qué es NDJSON, ver los campos,
cargar a pandas, comparar los 16 subreddits y construir un corpus limpio.

**Parte 2 - Avanzado** (secciones 10 a 12): soluciones de los 5 ejercicios de la
parte 1, y cómo leer los dumps históricos comprimidos `.zst` en streaming.

Ejecuta las celdas en orden con `Shift+Enter`.

**Requisitos:** `pandas`, `matplotlib`, `openpyxl` (ejercicio 5) y `zstandard`
(sección 11). Si te falta alguno:

```
pip install pandas matplotlib openpyxl zstandard
```

---
# Parte 1 - Fundamentos

## 1. Qué es un archivo NDJSON

NDJSON significa *Newline Delimited JSON*: un archivo de texto donde **cada línea es un
objeto JSON completo e independiente** (en nuestro caso, un post de Reddit).

A diferencia de un JSON normal (que es un solo objeto gigante y hay que cargarlo entero),
el NDJSON se puede procesar línea por línea, lo cual es ideal para archivos grandes:
nunca necesitas más memoria que la de una línea.

```
{"title": "post 1", "score": 10, ...}
{"title": "post 2", "score": 3, ...}
{"title": "post 3", "score": 7, ...}
```

Primero configuramos la ruta a la carpeta de datos y listamos qué hay dentro.

In [ ]:
from pathlib import Path
import json

# Ruta a la carpeta con los archivos NDJSON (ajústala si mueves los datos)
DATA_DIR = Path("/Users/ppizam/Library/CloudStorage/GoogleDrive-pizacapital@gmail.com"
                "/Other computers/My Mac RRG/data/thesis_reddit/raw/reddit"
                "/monthly 2026/2026-06-api-arctic-shift")

assert DATA_DIR.exists(), f"No encuentro la carpeta: {DATA_DIR}"

# glob busca archivos que cumplan un patrón; sorted() los ordena alfabéticamente
archivos = sorted(DATA_DIR.glob("*_submissions.ndjson"))

print(f"{len(archivos)} archivos encontrados:\n")
for f in archivos:
    mb = f.stat().st_size / 1_000_000   # tamaño en megabytes
    print(f"  {f.name:45s} {mb:8.2f} MB")

## 2. Ver una línea cruda

Antes de usar librerías, mira el archivo tal cual es: texto plano.
Abrimos uno y leemos solo la primera línea.

In [ ]:
# Usamos Vitards porque es el más pequeño (33 posts)
archivo = DATA_DIR / "Vitards_submissions.ndjson"

with open(archivo, encoding="utf-8") as f:
    primera_linea = f.readline()

print(f"Longitud de la línea: {len(primera_linea):,} caracteres\n")
print(primera_linea[:600] + " ...")   # solo los primeros 600 caracteres

Es JSON puro: pares `"campo": valor` separados por comas. Ilegible así, pero
perfecto para una máquina.

## 3. Convertir la línea a diccionario y ver los "encabezados"

`json.loads()` convierte el texto JSON en un **diccionario de Python**. Las *llaves*
del diccionario son el equivalente a los encabezados de columna de un Excel.

In [ ]:
post = json.loads(primera_linea)

print(f"Tipo del objeto: {type(post).__name__}")
print(f"Número de campos: {len(post)}\n")

# sorted(post.keys()) = lista alfabética de todos los campos ("encabezados")
campos = sorted(post.keys())
for i in range(0, len(campos), 3):          # imprimir en 3 columnas
    fila = campos[i:i+3]
    print("".join(f"{c:38s}" for c in fila))

Son ~100 campos, pero para análisis casi siempre bastan estos:

| Campo | Qué contiene |
|---|---|
| `id` | Identificador único del post |
| `title` | Título (siempre sobrevive, aunque borren el post) |
| `selftext` | Cuerpo del post - `"[removed]"` si lo borró moderación, `"[deleted]"` si lo borró el autor, vacío si es post de link/imagen |
| `author` | Usuario que publicó |
| `created_utc` | Fecha-hora de publicación en formato *Unix timestamp* (segundos desde 1970) |
| `score` | Upvotes netos |
| `num_comments` | Número de comentarios |
| `link_flair_text` | Etiqueta del post (DD, Discussion, News...) |
| `is_self` | `True` = post de texto; `False` = link o imagen |
| `removed_by_category` | Quién lo borró: `moderator`, `automod_filtered`, `reddit`, `deleted`... |
| `upvote_ratio` | Proporción de votos positivos |

Veamos esos campos en nuestro post de ejemplo:

In [ ]:
from datetime import datetime, timezone

CAMPOS_CLAVE = ["id", "title", "author", "created_utc", "score", "num_comments",
                "link_flair_text", "is_self", "removed_by_category"]

for campo in CAMPOS_CLAVE:
    print(f"{campo:22s} -> {post.get(campo)!r}")

# created_utc es un timestamp Unix; así se convierte a fecha legible:
fecha = datetime.fromtimestamp(post["created_utc"], tz=timezone.utc)
print(f"\ncreated_utc convertido -> {fecha:%Y-%m-%d %H:%M} UTC")

# selftext lo mostramos aparte porque puede ser largo
print(f"\nselftext (primeros 300 caracteres):\n{post['selftext'][:300]}")

> Tip: `post.get(campo)` es más seguro que `post[campo]` porque devuelve `None`
> en lugar de lanzar error si el campo no existe en algún registro.

## 4. Cargar un archivo completo

Ahora leemos todas las líneas de un archivo y las convertimos en una lista de
diccionarios. Encapsulamos la lógica en una función para reutilizarla.

In [ ]:
def cargar_ndjson(ruta):
    """Lee un archivo NDJSON y devuelve una lista de diccionarios (un post por línea)."""
    posts = []
    with open(ruta, encoding="utf-8") as f:
        for linea in f:
            linea = linea.strip()
            if linea:                      # ignorar líneas vacías por si acaso
                posts.append(json.loads(linea))
    return posts

posts = cargar_ndjson(DATA_DIR / "Vitards_submissions.ndjson")
print(f"Posts cargados: {len(posts)}")

# Con una lista de dicts ya puedes hacer preguntas con Python puro.
# Ejemplo: ¿cuántos posts tienen el cuerpo borrado?
borrados = sum(1 for p in posts if p["selftext"] in ("[removed]", "[deleted]"))
print(f"Con texto borrado: {borrados} de {len(posts)}")

## 5. Pasar a pandas

Para análisis serio conviene un `DataFrame`. La clave: **no cargues los ~100 campos**,
selecciona solo los que necesitas. Esto mantiene todo rápido y ligero incluso con el
archivo de wallstreetbets (11,782 posts).

In [ ]:
import pandas as pd

def ndjson_a_dataframe(ruta, campos=CAMPOS_CLAVE + ["selftext", "upvote_ratio"]):
    """Carga un NDJSON a DataFrame quedándose solo con los campos indicados."""
    filas = [{c: p.get(c) for c in campos} for p in cargar_ndjson(ruta)]
    df = pd.DataFrame(filas)
    # Convertir el timestamp Unix a fecha; .dt.date extrae solo el día
    df["fecha"] = pd.to_datetime(df["created_utc"], unit="s", utc=True)
    return df

df = ndjson_a_dataframe(DATA_DIR / "stocks_submissions.ndjson")
print(df.shape)      # (filas, columnas)
df.head(3)

In [ ]:
# info() = radiografía del DataFrame: tipos de dato y valores no nulos por columna
df.info()

In [ ]:
# describe() = estadísticas de las columnas numéricas
df[["score", "num_comments", "upvote_ratio"]].describe().round(2)

## 6. Análisis exploratorio básico

Tres herramientas de pandas resuelven el 80% de la exploración:

- `value_counts()` - tabla de frecuencias de una columna
- filtros booleanos - `df[condición]`
- `groupby()` / `resample()` - agrupar por categoría o por tiempo

In [ ]:
# ¿Quién borró los posts? value_counts() cuenta cada valor distinto
# dropna=False hace que también cuente los NaN (posts NO borrados)
print("removed_by_category:")
print(df["removed_by_category"].value_counts(dropna=False))

print("\nAutores más activos:")
print(df["author"].value_counts().head(8))

In [ ]:
# Filtros booleanos: la condición crea una serie True/False que indexa el DataFrame
mask_borrado = df["selftext"].isin(["[removed]", "[deleted]"])

print(f"Posts totales:        {len(df):>6,}")
print(f"Con texto borrado:    {mask_borrado.sum():>6,}  ({100*mask_borrado.mean():.1f}%)")
print(f"Con texto disponible: {(~mask_borrado).sum():>6,}")

# ~ invierte la máscara; .sample(3) toma 3 filas al azar de los que sí tienen texto
df.loc[~mask_borrado, ["fecha", "title", "score"]].sample(3, random_state=1)

In [ ]:
import matplotlib.pyplot as plt

# Posts por día: agrupamos por el día de la fecha y contamos
por_dia = df.groupby(df["fecha"].dt.date).size()

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(range(len(por_dia)), por_dia.values, color="#00684A", width=0.7)
ax.set_xticks(range(0, len(por_dia), 2))
ax.set_xticklabels([d.strftime("%d") for d in por_dia.index[::2]])
ax.set_title("r/stocks - posts por día, junio 2026", loc="left")
ax.set_xlabel("día del mes")
ax.spines[["top", "right"]].set_visible(False)   # quitar bordes que estorban
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 7. Recorrer los 16 subreddits y compararlos

El mismo análisis, en un bucle sobre todos los archivos. Construimos una tabla
resumen con una fila por subreddit - este es el diagnóstico general del dataset.

In [ ]:
resumen = []
for ruta in archivos:
    sub = ruta.name.replace("_submissions.ndjson", "")
    d = ndjson_a_dataframe(ruta)
    borrado = d["selftext"].isin(["[removed]", "[deleted]"])
    resumen.append({
        "subreddit": sub,
        "posts": len(d),
        "primer_post": d["fecha"].min().strftime("%m-%d"),
        "ultimo_post": d["fecha"].max().strftime("%m-%d"),
        "pct_borrado": round(100 * borrado.mean(), 1),
        "pct_link": round(100 * (~d["is_self"]).mean(), 1),   # posts de link/imagen
        "pct_texto_util": round(100 * (d["is_self"] & ~borrado).mean(), 1),
        "score_mediano": d["score"].median(),
    })

tabla = pd.DataFrame(resumen).sort_values("posts", ascending=False).reset_index(drop=True)
tabla

In [ ]:
# Visualizar el tamaño de cada subreddit (barras horizontales, ordenadas)
t = tabla.sort_values("posts")

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(t["subreddit"], t["posts"], color="#00684A", height=0.65)
for y, v in enumerate(t["posts"]):
    ax.text(v, y, f" {v:,}", va="center", fontsize=9, color="#444444")
ax.set_title("Posts por subreddit - junio 2026", loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.margins(x=0.12)
plt.tight_layout()
plt.show()

## 8. Construir un corpus limpio

Para análisis de sentimiento te conviene quitar el ruido. Criterios típicos:

1. Fuera posts de `AutoModerator` (hilos automáticos semanales)
2. Fuera posts con cuerpo `[removed]` / `[deleted]` (si necesitas el selftext)
3. Combinar `title` + `selftext` en un solo campo de texto

Nota: si tu unidad de análisis es el **título** (que siempre sobrevive), puedes
saltarte el criterio 2 y conservar mucho más corpus.

In [ ]:
def corpus_limpio(ruta):
    """Devuelve un DataFrame filtrado con una columna 'texto' lista para NLP."""
    d = ndjson_a_dataframe(ruta)
    d = d[d["author"] != "AutoModerator"]
    d = d[~d["selftext"].isin(["[removed]", "[deleted]"])]
    d["selftext"] = d["selftext"].fillna("")
    d["texto"] = (d["title"] + " " + d["selftext"]).str.strip()
    return d.reset_index(drop=True)

limpio = corpus_limpio(DATA_DIR / "pennystocks_submissions.ndjson")
original = len(ndjson_a_dataframe(DATA_DIR / "pennystocks_submissions.ndjson"))
print(f"pennystocks: {original:,} posts -> {len(limpio):,} tras limpieza")

limpio[["fecha", "texto", "score"]].sample(3, random_state=2)

## 9. Ejercicios para practicar

1. Carga `Superstonk_submissions.ndjson` y saca sus 10 flairs más comunes con `value_counts()`.
2. ¿Qué subreddit tiene el post con más comentarios de todo junio?
3. Grafica los posts por día de wallstreetbets. ¿Hay picos? ¿Coinciden con algún evento?
4. En la tabla resumen, agrega una columna `pct_automod` con el porcentaje de posts
   de AutoModerator.
5. Exporta la tabla resumen a Excel con `to_excel`.

Las soluciones están en la parte 2, pero intenta resolverlos primero por tu cuenta.

---
# Parte 2 - Ejercicios resueltos y dumps .zst

## 10. Soluciones de los ejercicios

### Ejercicio 1 - Los 10 flairs más comunes de Superstonk

Reutilizamos `ndjson_a_dataframe` (por eso conviene encapsular en funciones).

In [ ]:
ss = ndjson_a_dataframe(DATA_DIR / "Superstonk_submissions.ndjson")

# dropna=False para ver también cuántos posts NO tienen flair
ss["link_flair_text"].value_counts(dropna=False).head(10)

### Ejercicio 2 - El post con más comentarios de todo junio

Estrategia: recorrer los 16 archivos y, en cada uno, quedarnos con la fila cuyo
`num_comments` es máximo (`idxmax()` devuelve el índice de esa fila). Al final
comparamos los 16 campeones.

In [ ]:
campeones = []
for ruta in archivos:
    d = ndjson_a_dataframe(ruta)
    fila = d.loc[d["num_comments"].idxmax()]        # la fila con el máximo
    campeones.append({
        "subreddit": ruta.name.replace("_submissions.ndjson", ""),
        "num_comments": fila["num_comments"],
        "title": fila["title"][:60],
        "author": fila["author"],
        "fecha": fila["fecha"].strftime("%m-%d"),
    })

camp = pd.DataFrame(campeones).sort_values("num_comments", ascending=False).reset_index(drop=True)
print(f"Ganador absoluto: r/{camp.iloc[0]['subreddit']} con {camp.iloc[0]['num_comments']:,} comentarios")
camp.head(8)

### Ejercicio 3 - Posts por día en wallstreetbets

El mismo patrón de la sección 6, cambiando el archivo. Ojo: los hilos diarios de
discusión concentran la conversación, así que los picos de *posts* suelen reflejar
eventos de mercado que disparan publicaciones individuales.

In [ ]:
wsb = ndjson_a_dataframe(DATA_DIR / "wallstreetbets_submissions.ndjson")
wsb_dia = wsb.groupby(wsb["fecha"].dt.date).size()

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(range(len(wsb_dia)), wsb_dia.values, color="#00684A", width=0.7)
ax.set_xticks(range(0, len(wsb_dia), 2))
ax.set_xticklabels([d.strftime("%d") for d in wsb_dia.index[::2]])
ax.set_title("r/wallstreetbets - posts por día, junio 2026", loc="left")
ax.set_xlabel("día del mes")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

# Los 3 días con más actividad, para investigar qué pasó en el mercado
print("Días pico:")
print(wsb_dia.sort_values(ascending=False).head(3))

Para interpretar los picos vale la pena cruzarlos con el calendario de mercado
(reportes de resultados, datos de inflación, decisiones de la Fed). Ese cruce es
justo el tipo de análisis que conecta con la tesis.

### Ejercicio 4 - Agregar `pct_automod` a la tabla resumen

Misma construcción de la sección 7, con una línea extra por subreddit.

In [ ]:
resumen2 = []
for ruta in archivos:
    sub = ruta.name.replace("_submissions.ndjson", "")
    d = ndjson_a_dataframe(ruta)
    borrado = d["selftext"].isin(["[removed]", "[deleted]"])
    resumen2.append({
        "subreddit": sub,
        "posts": len(d),
        "pct_borrado": round(100 * borrado.mean(), 1),
        "pct_link": round(100 * (~d["is_self"]).mean(), 1),
        "pct_texto_util": round(100 * (d["is_self"] & ~borrado).mean(), 1),
        # La línea nueva: proporción de posts publicados por AutoModerator
        "pct_automod": round(100 * (d["author"] == "AutoModerator").mean(), 1),
    })

tabla_v2 = pd.DataFrame(resumen2).sort_values("posts", ascending=False).reset_index(drop=True)
tabla_v2

### Ejercicio 5 - Exportar la tabla a Excel

`to_excel` necesita la librería `openpyxl` (`pip install openpyxl`). El archivo se
guarda en la carpeta donde corre el notebook; con `Path.cwd()` verificamos cuál es.

In [ ]:
salida = Path.cwd() / "resumen_junio_2026.xlsx"
tabla_v2.to_excel(salida, index=False, sheet_name="junio 2026")
print(f"Guardado en: {salida}")
print(f"Existe: {salida.exists()}  |  {salida.stat().st_size:,} bytes")

## 11. Leer dumps históricos .zst en streaming

Tus carpetas `subreddits23` contienen los dumps históricos de Pushshift, comprimidos
con **zstandard** (`.zst`). Dos diferencias importantes contra los NDJSON de junio:

1. **Van comprimidos** - `Vitards_submissions.zst` pesa 10.5 MB pero descomprimido
   es mucho más grande. No conviene descomprimirlo a disco: se lee *en streaming*,
   descomprimiendo en memoria línea por línea.
2. **Necesitan un parámetro especial** - estos dumps fueron creados con una "ventana"
   de compresión de hasta 2 GB. Si no le pasas `max_window_size=2**31` al
   descompresor, zstandard lanza un error de ventana excedida.

Instala la librería si te falta: `pip install zstandard`.

Nota sobre la ruta: uso la copia local de `Rodrigo Ramos/data/thesis_reddit` porque
los `.zst` de la copia de Google Drive están en la nube sin descargar. Si descargas
los de Drive, solo cambia `ZST_PATH`.

In [ ]:
import io
import zstandard as zstd

ZST_PATH = Path("/Users/ppizam/Claude/Master Thesis/Rodrigo Ramos/data/thesis_reddit"
                "/raw/reddit/subreddits23/Vitards_submissions.zst")
assert ZST_PATH.exists(), f"No encuentro el archivo: {ZST_PATH}"

def leer_zst(ruta):
    """Generador: entrega un dict por línea del .zst, sin cargar todo a memoria.

    'yield' pausa la función y entrega un post a la vez - por eso puede procesar
    archivos de gigabytes con memoria mínima.
    """
    dctx = zstd.ZstdDecompressor(max_window_size=2**31)   # ventana de 2 GB
    with open(ruta, "rb") as fh:                          # "rb" = binario
        with dctx.stream_reader(fh) as reader:
            texto = io.TextIOWrapper(reader, encoding="utf-8")
            for linea in texto:
                yield json.loads(linea)

# Probamos: leer solo el primer post y ver sus campos
primero = next(leer_zst(ZST_PATH))
print(f"Campos del dump histórico: {len(primero)}")
print(f"Primer post: {datetime.fromtimestamp(int(primero['created_utc']), tz=timezone.utc):%Y-%m-%d} - {primero['title'][:60]}")

Fíjate que el dump histórico tiene ~82 campos contra ~100 del API de Arctic Shift:
son capturas de épocas y metodologías distintas, así que **antes de combinar fuentes
verifica que los campos que usarás existan en ambas**.

Ahora un barrido completo del archivo: contar posts y medir el rango de fechas,
sin cargar nada completo a memoria.

In [ ]:
n = 0
fecha_min = None
fecha_max = None

for p in leer_zst(ZST_PATH):
    n += 1
    ts = int(p.get("created_utc") or 0)
    if ts:
        fecha_min = ts if fecha_min is None or ts < fecha_min else fecha_min
        fecha_max = ts if fecha_max is None or ts > fecha_max else fecha_max

a_fecha = lambda ts: datetime.fromtimestamp(ts, tz=timezone.utc).date()
print(f"Posts en el dump: {n:,}")
print(f"Rango: {a_fecha(fecha_min)} a {a_fecha(fecha_max)}")

Este dump de Vitards cubre desde 2021 (la era del *squeeze* de aceros que dio origen
al subreddit) hasta fines de 2023.

El patrón más útil: **filtrar mientras se lee**, quedándote solo con lo que te
interesa en un DataFrame pequeño. Ejemplo: los posts de 2021 con más de 100 de score.

In [ ]:
seleccion = []
for p in leer_zst(ZST_PATH):
    ts = int(p.get("created_utc") or 0)
    anio = datetime.fromtimestamp(ts, tz=timezone.utc).year if ts else None
    score = int(p.get("score") or 0)
    if anio == 2021 and score > 100:               # el filtro ocurre durante la lectura
        seleccion.append({
            "fecha": datetime.fromtimestamp(ts, tz=timezone.utc),
            "title": p.get("title"),
            "score": score,
            "num_comments": p.get("num_comments"),
        })

hist = pd.DataFrame(seleccion).sort_values("score", ascending=False).reset_index(drop=True)
print(f"Posts de 2021 con score > 100: {len(hist):,}")
hist.head(10)

## 12. Qué sigue

Con lo que ya sabes puedes replicar todo el flujo en cualquier fuente del proyecto:

1. **Otros meses de 2026** - aplica las secciones 5 a 8 a `raw/monthly/2026-01`
   a `2026-05` (misma estructura NDJSON).
2. **Dumps .zst grandes** - `wallstreetbets_comments.zst` pesa 6.6 GB comprimido;
   el generador `leer_zst` lo procesa igual, solo tomará más tiempo. Filtra mientras
   lees y nunca lo cargues completo.
3. **Comentarios de junio 2026** - no están en la carpeta del API; habría que
   descargarlos con un script tipo `fetch_june.py` apuntando al endpoint de
   comentarios de Arctic Shift. Esa puede ser la parte 3.

**Ejercicios nuevos para practicar:**

1. Cuenta cuántos posts del dump histórico de Vitards son de cada año (pista:
   un `Counter` de `collections` y el año de `created_utc`).
2. Del dump histórico, extrae los posts de enero 2021 y calcula su score mediano.
3. Compara el % de `[removed]` del dump histórico contra el 39.4% de junio 2026.
   ¿La moderación de Vitards era igual de agresiva en 2021-2023?